# JAX-CrossCat: Introduction Tutorial

This notebook walks through the core workflow of JAX-CrossCat — from generating data, to running inference, to querying the posterior.

**What is CrossCat?** A Bayesian model for heterogeneous tabular data that simultaneously discovers:
1. **Which columns are related** (grouping columns into "views")
2. **How rows cluster within each group** (independent clustering per view)

**Requirements:** `pip install -e .` or `uv sync --extra dev` from the repo root.

## 1. Setup and Imports

In [ ]:
import jax
import jax.numpy as jnp

from crosscat import (
    collect_diagnostics,
    dependence_matrix,
    generate_crosscat_data,
    guess_column_types,
    impute_and_confidence,
    initialize,
    log_joint,
    mutual_information,
    predictive_anomalousness,
    predictive_probability,
    predictive_sample,
    row_similarity,
    validate_state,
)
from crosscat.packed import pack_state, packed_gibbs_sweep, unpack_state
from crosscat.types import ColumnType

print(f"JAX version: {jax.__version__}")
print(f"Devices: {jax.devices()}")

## 2. Generate Synthetic Data

We'll create a dataset with known structure so we can verify that CrossCat recovers it.

Our data has:
- **200 rows** (observations)
- **6 columns** of mixed types: CONTINUOUS, CATEGORICAL, BINARY, ORDINAL, CYCLIC, CONTINUOUS
- **2 views** (column groups) with **2 clusters** each

In [ ]:
key = jax.random.key(42)

column_types = [
    ColumnType.CONTINUOUS,    # e.g., salary
    ColumnType.CATEGORICAL,   # e.g., department (0, 1, 2, ...)
    ColumnType.BINARY,        # e.g., is_remote (0 or 1)
    ColumnType.ORDINAL,       # e.g., rating (1, 2, 3, 4, 5)
    ColumnType.CYCLIC,        # e.g., wind direction (0 to 2*pi)
    ColumnType.CONTINUOUS,    # e.g., years_experience
]

result = generate_crosscat_data(
    key, n_rows=200, column_types=column_types,
    n_views=2, n_clusters=2, cluster_separation=5.0,
)

data = result["data"]
true_col_assignments = result["true_column_assignments"]

print(f"Data shape: {data.shape}")
print(f"Column types: {[ct.value for ct in column_types]}")
print(f"True column-to-view assignments: {true_col_assignments}")
print(f"\nFirst 5 rows:\n{data[:5]}")

## 3. Auto-Detect Column Types

If you're loading real data (e.g., from CSV), you can let CrossCat guess the column types:

In [ ]:
# Auto-detect types from the data itself
guessed_types = guess_column_types(data)
for i, (true_t, guessed_t) in enumerate(zip(column_types, guessed_types, strict=True)):
    match = "OK" if true_t == guessed_t else "MISMATCH"
    print(f"  Col {i}: true={true_t.value:12s}  guessed={guessed_t.value:12s}  {match}")

# Note: ORDINAL and CYCLIC can't be auto-detected (they look like CATEGORICAL/CONTINUOUS).
# Always specify these manually for best results.

## 4. Initialize the Model

`initialize()` creates a `CrossCatState` by sampling cluster assignments from the CRP prior.

In [ ]:
key, init_key = jax.random.split(key)
state = initialize(init_key, data, column_types)

print(f"Number of views discovered: {state.n_views}")
print(f"Number of rows: {state.n_rows}")
print(f"Number of columns: {state.n_cols}")
print(f"Column assignments: {state.column_assignments}")
for i, view in enumerate(state.views):
    n_clusters = len(set(int(x) for x in view.row_assignments))
    print(f"  View {i}: {n_clusters} clusters, columns {[j for j, v in enumerate(state.column_assignments) if v == i]}")

# Validate state consistency
errors = validate_state(state, data)
print(f"\nValidation errors: {len(errors)}")

## 5. Run Inference (Packed Path)

The **packed path** converts state to fixed-size JAX arrays for JIT compilation — this is 10-100x faster than the unpacked path.

**Workflow:** `CrossCatState` -> `pack_state()` -> `packed_gibbs_sweep()` -> `unpack_state()` -> query

In [ ]:
%%time

# Pack state for GPU/JIT
packed = pack_state(state)

# Run 50 Gibbs sweeps (first call triggers JIT compilation)
key, sweep_key = jax.random.split(key)
packed = packed_gibbs_sweep(sweep_key, packed, data, n_sweeps=50)

# Unpack back to query-friendly state
state = unpack_state(packed, column_types, data=data)

score = float(log_joint(state, data))
print(f"\nLog joint after 50 sweeps: {score:.2f}")
print(f"Views discovered: {state.n_views}")
print(f"Learned column assignments: {state.column_assignments}")
print(f"True column assignments:    {true_col_assignments}")

## 6. Diagnostics

Check convergence by looking at log joint and diagnostics over time:

In [ ]:
diag = collect_diagnostics(state, data)
for k, v in diag.items():
    print(f"  {k}: {v}")

## 7. Query the Posterior

Now the fun part — asking questions about the data.

### 7a. Dependence Discovery

*"Which columns are related?"* The dependence matrix (Z-matrix) shows pairwise probability that columns share a view:

In [ ]:
z_matrix = dependence_matrix([state])
print("Dependence matrix (Z-matrix):")
print("  Columns in the SAME view should have values close to 1.0")
print("  Columns in DIFFERENT views should have values close to 0.0\n")

col_labels = ["CONT", "CAT", "BIN", "ORD", "CYC", "CONT"]
header = "       " + "  ".join(f"{label:>5s}" for label in col_labels)
print(header)
for i, row in enumerate(z_matrix):
    vals = "  ".join(f"{float(v):5.2f}" for v in row)
    print(f"  {col_labels[i]:>5s} {vals}")

### 7b. Mutual Information

*"How much does column 0 tell us about column 5?"*

In [ ]:
key, mi_key = jax.random.split(key)

# MI between columns in the same view (should be higher)
mi_same, linfoot_same = mutual_information([state], col_i=0, col_j=5, rng_key=mi_key)
print(f"MI(col_0, col_5) [likely same view]:      MI={float(mi_same):.4f}, Linfoot={float(linfoot_same):.4f}")

# MI between columns in different views (should be near 0)
mi_diff, linfoot_diff = mutual_information([state], col_i=0, col_j=4, rng_key=mi_key)
print(f"MI(col_0, col_4) [likely different views]: MI={float(mi_diff):.4f}, Linfoot={float(linfoot_diff):.4f}")

### 7c. Predictive Probability

*"How likely is a specific value for row 0, column 0?"*

In [ ]:
# How likely is the observed value?
observed_val = float(data[0, 0])
logp = predictive_probability(
    state, data, query_cols=[0], query_vals=jnp.array([observed_val]), row_id=0,
)
print(f"Observed value: {observed_val:.3f}")
print(f"Log predictive probability: {float(logp):.4f}")

# Compare with an unlikely value far from the data
unlikely_val = 100.0
logp_unlikely = predictive_probability(
    state, data, query_cols=[0], query_vals=jnp.array([unlikely_val]), row_id=0,
)
print(f"\nUnlikely value: {unlikely_val}")
print(f"Log predictive probability: {float(logp_unlikely):.4f}")

### 7d. Predictive Sampling

*"Generate plausible values for column 0, conditioned on the context of row 0:"*

In [ ]:
key, sample_key = jax.random.split(key)

# Draw 10 samples from the posterior predictive for column 0, given row 0 context
samples = predictive_sample(sample_key, state, data, query_cols=[0], row_id=0, n_samples=10)
print(f"Observed value:     {float(data[0, 0]):.3f}")
print(f"Predictive samples: {[f'{float(s):.3f}' for s in samples[:, 0]]}")

### 7e. Imputation

*"What's the best guess for a missing value, and how confident are we?"*

In [ ]:
key, impute_key = jax.random.split(key)
value, confidence = impute_and_confidence(impute_key, state, data, query_col=0, row_id=0)
print(f"True value:    {float(data[0, 0]):.3f}")
print(f"Imputed value: {float(value):.3f}")
print(f"Confidence:    {float(confidence):.3f}")

### 7f. Anomaly Detection

*"Which rows are unusual?"*

In [ ]:
key, anom_key = jax.random.split(key)

# Score a few rows
scores = []
for row_id in range(min(20, data.shape[0])):
    s = predictive_anomalousness(
        jax.random.fold_in(anom_key, row_id), state, data, query_row=row_id,
    )
    scores.append((row_id, float(s)))

scores.sort(key=lambda x: x[1], reverse=True)
print("Top 5 most anomalous rows:")
for row_id, score in scores[:5]:
    print(f"  Row {row_id:3d}: anomaly score = {score:.4f}")

### 7g. Row Similarity

*"How similar are two rows across the learned structure?"*

In [ ]:
# Row similarity: probability that two rows are in the same cluster
sim_0_1 = row_similarity([state], row_a=0, row_b=1)
sim_0_100 = row_similarity([state], row_a=0, row_b=100)
sim_0_0 = row_similarity([state], row_a=0, row_b=0)

print(f"Similarity(row 0, row 1):   {float(sim_0_1):.3f}")
print(f"Similarity(row 0, row 100): {float(sim_0_100):.3f}")
print(f"Similarity(row 0, row 0):   {float(sim_0_0):.3f}  (self = 1.0)")

## 8. Saving and Loading

Save the model for later use:

In [ ]:
from crosscat import load_packed_state, save_packed_state

# Save
packed = pack_state(state)
save_packed_state(packed, "/tmp/intro_tutorial_model", column_types=column_types)
print("Model saved to /tmp/intro_tutorial_model/")

# Load
packed_loaded, loaded_types = load_packed_state("/tmp/intro_tutorial_model")
state_loaded = unpack_state(packed_loaded, loaded_types, data=data)
print(f"Model loaded: {state_loaded.n_views} views, {state_loaded.n_cols} columns")
print(f"Log joint matches: {jnp.allclose(log_joint(state, data), log_joint(state_loaded, data))}")

## Next Steps

- **More sweeps**: 50-200 sweeps is typical; watch `log_joint` for convergence
- **Multi-chain**: Run 4+ chains with `initialize(key, data, col_types, n_chains=4)` and pick the best by `log_joint`
- **Real data**: Use `read_csv()` and `guess_column_types()` to load your own CSV files
- **GPU acceleration**: See `docs/guides/gpu-packed.md` for JIT compilation and XLA caching
- **Constraints**: Force columns together/apart with `ensure_col_dep_constraints()`
- **Online learning**: Add new rows with `packed_insert_rows()`

See the full API reference in `docs/api/` and feature guides in `docs/guides/`.